# Exercises XP — BERT Toolbox

**Course:** Developers Institute  **Week 7 - Day 3**  
**Author:** Alex Goldbaum

Six exercises covering the practical BERT workflow:
1. Tokenization with `bert-base-uncased` and inspection of special tokens.
2. Sentiment analysis via the Hugging Face `pipeline`.
3. A from-scratch `BERTSentimentAnalyzer` class with full control of
   tokenizer + model + post-processing.
4. A `BERTNamedEntityRecognizer` class for token-classification (NER).
5. BERT vs GPT — comparison table + reflection.
6. BERT's role in Retrieval-Augmented Generation systems.


## Setup


In [ ]:
%pip install -qU transformers==4.* torch scikit-learn


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
import torch
from torch.nn.functional import softmax

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)


## Exercise 1 — Tokenization with BERT

BERT does not consume raw text. It splits each sentence into **WordPiece**
subword units, wraps them with `[CLS]` (start) and `[SEP]` (separator), and
maps every token to an integer ID. Optional padding (`[PAD]`) brings every
input to the same length so we can batch them.


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

sentence = 'BERT tokenizes sentences using subword units like ##ization.'

# Tokens before any special markers (just to see the WordPiece split)
raw_tokens = tokenizer.tokenize(sentence)
print('Raw tokens:', raw_tokens)

# Full encoding with special tokens + padding + truncation
enc = tokenizer(
    sentence,
    add_special_tokens=True,
    padding='max_length',
    truncation=True,
    max_length=24,
    return_tensors='pt',
)

ids = enc['input_ids'][0].tolist()
tokens_with_special = tokenizer.convert_ids_to_tokens(ids)

print('\nTokens with special markers + padding:')
for token, tid in zip(tokens_with_special, ids):
    note = ''
    if token in tokenizer.all_special_tokens:
        note = '  <-- special'
    print(f'  id={tid:>6}  token={token!r}{note}')

print('\nattention_mask:', enc['attention_mask'][0].tolist())


**What we see.**
- BERT prepends `[CLS]` (id `101`) and appends `[SEP]` (id `102`).
- The rare token `tokenization` is split into the subwords `token` + `##ization`.
- `[PAD]` (id `0`) fills the tail to reach `max_length=24`.
- `attention_mask` is `1` for real tokens and `0` for the padding — this tells
  the model to ignore the padded positions when computing self-attention.


## Exercise 2 — Sentiment Analysis with the `pipeline`

The `pipeline` API hides tokenizer + model + post-processing behind a single
callable. We use the canonical `distilbert-base-uncased-finetuned-sst-2-english`
checkpoint, already fine-tuned on the SST-2 movie-review sentiment task.


In [ ]:
from transformers import pipeline

sentiment = pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    device=0 if torch.cuda.is_available() else -1,
)

samples = [
    'I absolutely loved this movie, the cinematography was breathtaking!',
    'Worst service of my life. Slow, rude, and overpriced.',
    'It was okay, nothing special.',
    'Best customer support I have ever experienced. 10/10 would recommend.',
]

results = sentiment(samples)
for text, res in zip(samples, results):
    print(f'>>> {text}')
    print(f'   label={res["label"]}  confidence={res["score"]:.4f}')
    print()


## Exercise 3 — Custom `BERTSentimentAnalyzer` Class

Wrapping the same checkpoint in our own class. We expose preprocessing,
tokenization and post-processing as separate methods so each step can be
tuned, logged or unit-tested.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification


class BERTSentimentAnalyzer:
    """Hand-rolled sentiment analyzer using a fine-tuned BERT/DistilBERT checkpoint."""

    def __init__(
        self,
        model_name: str = 'distilbert-base-uncased-finetuned-sst-2-english',
        device: torch.device = None,
        max_length: int = 128,
    ):
        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name).to(self.device).eval()
        self.id2label = self.model.config.id2label
        self.max_length = max_length

    @staticmethod
    def _clean(text: str) -> str:
        """Lightweight preprocessing — strip control chars and excessive whitespace."""
        text = text.replace('\r', ' ').replace('\n', ' ')
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def _tokenize(self, text: str):
        return self.tokenizer(
            text,
            truncation=True, padding='max_length', max_length=self.max_length,
            return_tensors='pt',
        ).to(self.device)

    def predict(self, text: str) -> dict:
        clean = self._clean(text)
        enc = self._tokenize(clean)
        with torch.no_grad():
            logits = self.model(**enc).logits
        probs = softmax(logits, dim=-1)[0].cpu().numpy()
        pred_id = int(probs.argmax())
        return {
            'text': text,
            'clean_text': clean,
            'label': self.id2label[pred_id],
            'confidence': float(round(probs[pred_id], 4)),
            'all_scores': {self.id2label[i]: float(round(p, 4)) for i, p in enumerate(probs)},
        }

    def predict_batch(self, texts):
        return [self.predict(t) for t in texts]


analyzer = BERTSentimentAnalyzer()

test_texts = [
    'I cannot recommend this place enough — the service was outstanding.',
    'Honestly disappointed. Long wait, cold food.',
    'The product arrived broken AND the support team blamed me!',
    'A pleasant experience overall — would come back.',
    'Mediocre. Not bad, not great.',
]
for r in analyzer.predict_batch(test_texts):
    print(f">>> {r['text']}")
    print(f"    label={r['label']}  confidence={r['confidence']}  scores={r['all_scores']}")
    print()


## Exercise 4 — Custom `BERTNamedEntityRecognizer` Class

We use a BERT-base checkpoint fine-tuned on the CoNLL-2003 NER task with the
**B-I-O** scheme: `B-PER`, `I-PER`, `B-ORG`, `I-ORG`, `B-LOC`, `I-LOC`,
`B-MISC`, `I-MISC`, and `O` (outside). Our class tokenizes with
`is_split_into_words=False`, predicts per-token labels, and re-merges
subword pieces (`Hugging`, `##Face`) back into spans (`Hugging Face → ORG`).


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification


class BERTNamedEntityRecognizer:
    """Identify PER/ORG/LOC/MISC entities in free text using a BERT NER checkpoint."""

    def __init__(
        self,
        model_name: str = 'dslim/bert-base-NER',
        device: torch.device = None,
        max_length: int = 128,
    ):
        self.device = device or (torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu'))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name).to(self.device).eval()
        self.id2label = self.model.config.id2label
        self.max_length = max_length

    def recognize(self, text: str):
        enc = self.tokenizer(
            text,
            truncation=True, max_length=self.max_length,
            return_tensors='pt', return_offsets_mapping=True,
        )
        offsets = enc.pop('offset_mapping')[0].tolist()
        enc = {k: v.to(self.device) for k, v in enc.items()}

        with torch.no_grad():
            logits = self.model(**enc).logits
        preds = logits.argmax(dim=-1)[0].cpu().tolist()
        tokens = self.tokenizer.convert_ids_to_tokens(enc['input_ids'][0])

        # Merge consecutive B-/I- tags into entity spans
        entities, current = [], None
        for tok, pid, (start, end) in zip(tokens, preds, offsets):
            label = self.id2label[pid]
            # Skip special tokens
            if tok in self.tokenizer.all_special_tokens:
                if current is not None:
                    entities.append(current); current = None
                continue
            if label == 'O':
                if current is not None:
                    entities.append(current); current = None
                continue
            tag, kind = label.split('-', 1)  # 'B' or 'I', and 'PER'/'ORG'/'LOC'/'MISC'
            if tag == 'B' or current is None or current['type'] != kind:
                if current is not None:
                    entities.append(current)
                current = {'type': kind, 'start': start, 'end': end, 'tokens': [tok]}
            else:  # I- continuing the same entity
                current['end'] = end
                current['tokens'].append(tok)
        if current is not None:
            entities.append(current)

        # Attach the substring from the original text using offsets
        for ent in entities:
            ent['text'] = text[ent['start']:ent['end']]
        return entities


ner = BERTNamedEntityRecognizer()

ner_samples = [
    'Apple was founded in Cupertino by Steve Jobs and Steve Wozniak in 1976.',
    'Angela Merkel met with Emmanuel Macron in Paris last Tuesday.',
    'OpenAI, based in San Francisco, partnered with Microsoft to deploy GPT models.',
    'I traveled from Buenos Aires to Tokyo via Doha last summer.',
]

for s in ner_samples:
    print(f'>>> {s}')
    for ent in ner.recognize(s):
        print(f"   [{ent['type']:>4}] {ent['text']}  ({ent['start']}-{ent['end']})")
    print()


## Exercise 5 — BERT vs GPT

### Comparison table

| Aspect | **BERT** | **GPT** |
|---|---|---|
| **Architecture** | Transformer **encoder only** | Transformer **decoder only** |
| **Attention pattern** | Bidirectional — every token sees every other token | Causal (left-to-right) — each token sees only earlier tokens |
| **Pre-training objective** | Masked Language Modeling (predict masked tokens) + (originally) NSP | Causal Language Modeling (predict the next token) |
| **Primary purpose** | **Understanding** — produce a rich representation of an input | **Generation** — produce fluent next-token continuations |
| **Output shape** | One vector per token; the `[CLS]` representation drives classification | One generated token at a time, autoregressive |
| **Common use cases** | Classification, sentiment, NER, QA span extraction, embeddings for retrieval | Chatbots, code completion, summarization, story / dialogue generation |
| **Strengths** | Strong text understanding; small fine-tuning datasets work well; great for representation learning | Strong free-form generation; zero/few-shot capabilities; scales smoothly to 100B+ parameters |
| **Weaknesses** | Cannot generate text directly (no autoregressive decoder); needs labeled data per downstream task | Less efficient than encoders for pure classification; can hallucinate; expensive to serve at scale |
| **Iconic checkpoints** | BERT-base/large, RoBERTa, DistilBERT, ALBERT, ELECTRA, XLM-RoBERTa | GPT-2, GPT-3, GPT-4, LLaMA, Mistral, Claude (architecture family) |

### Reflection

BERT and GPT solve **different halves** of the language problem. BERT reads
and *understands*; GPT writes and *generates*. They are often complementary:
in a search-and-answer product, a BERT-style encoder retrieves the relevant
context and a GPT-style decoder writes the answer (the RAG pattern in
Exercise 6). Choose by the shape of the task: a fixed label or span at the
output → encoder-only; a free-form text continuation at the output →
decoder-only; transforming one sequence into another sequence → an
encoder-decoder like T5 or mBART.


## Exercise 6 — BERT in Retrieval-Augmented Generation (RAG)

**What RAG is.** A pattern where a generative model (typically a decoder-only
LLM like GPT) is *not* asked to remember everything. Instead, at inference
time, a retrieval system fetches the most relevant external documents and
the LLM is prompted to answer **using those documents as context**. This
gives access to fresh / private / proprietary knowledge, reduces
hallucination, and makes answers auditable (the sources are explicit).

**BERT's role: the retriever.** BERT-family encoders are the workhorse of the
retrieval step.

1. **Embedding documents (offline).** For each document in the corpus, we
   encode it with a BERT-style model — sometimes a **bi-encoder** (Sentence-
   BERT, E5, BGE) that mean-pools the encoder output into a fixed
   `d`-dimensional vector. We store every document vector in a **vector
   database** (FAISS, pgvector, Pinecone, Qdrant, Weaviate, …).
2. **Embedding the user query (online).** The same BERT model encodes the
   incoming question to the same `d`-dim space.
3. **Nearest-neighbour search.** The vector DB returns the top-K documents
   whose embeddings have the highest cosine similarity to the query
   embedding. Approximate nearest-neighbour algorithms (HNSW, IVF) make this
   sub-millisecond at corpus sizes of millions of documents.
4. **Optional re-ranking.** A more expensive **cross-encoder** (BERT reading
   the query and one candidate at a time) re-ranks the top-K for a better
   final ordering.
5. **Generation.** The top-K passages are pasted into the LLM prompt with the
   user question, and GPT (or another decoder model) generates the final
   answer.

**Why BERT-style embeddings are good at this.** Pre-training on huge corpora
with MLM teaches the encoder to map similar meanings to nearby vectors —
exactly the property semantic search needs. Bi-encoder versions are fine-
tuned on retrieval pairs (`anchor → positive` pairs from natural questions,
MS MARCO, etc.) to sharpen this further.

### Concrete example

User question: *"What benefits does the company offer for remote work?"*

1. The query is encoded by `sentence-transformers/all-MiniLM-L6-v2` (a small
   distilled BERT) into a 384-dim vector.
2. The vector DB returns the top-5 chunks from the HR handbook whose
   embeddings are closest to the query vector — e.g., the *Remote Work
   Allowance* section and the *Home Office Equipment* section.
3. A prompt is built:

```
Answer the question using only the context below.
If the answer is not in the context, say you do not know.

Context:
{top-5 retrieved chunks}

Question:
What benefits does the company offer for remote work?
```

4. GPT-4 (or Claude or LLaMA) generates the answer grounded in the retrieved
   passages — and we can show the user the exact source paragraphs as
   citations.

**Without BERT-style retrieval**, the LLM would have to *memorize* the HR
handbook during training (impractical and stale) or guess (hallucination).
With BERT as the retriever and GPT as the generator, RAG gives us fresh,
grounded, citeable answers — which is why almost every production
"AI assistant" today is a RAG system under the hood.


## Summary

- We saw how BERT turns text into IDs with WordPiece + `[CLS]/[SEP]/[PAD]`.
- A 2-line Hugging Face `pipeline` already gives us state-of-the-art sentiment;
  a custom class wrapping `AutoTokenizer` + `AutoModelForSequenceClassification`
  gives us the same result with full control over each step.
- A different `AutoModelForTokenClassification` checkpoint plus a small
  B-I-O re-merger turns BERT into a named-entity recognizer.
- BERT (encoder) and GPT (decoder) are complementary; pick the one whose
  output shape matches your task — or compose them in a RAG system, where
  BERT-style encoders retrieve and GPT generates the final answer.
